# DeepMzyme Training Notebook

This notebook runs DeepMzyme training through the existing command-line interface in `src/train.py`. It supports two runtime modes:

- `local`: Ubuntu / PyCharm / Jupyter from a local checkout.
- `colab`: Google Colab with Google Drive, Colab dependency setup, and `/content` runtime paths.

The notebook does not duplicate training logic. It builds a CLI command, runs it only when you explicitly enable training, then calls `src/report_runs.py` to create a comparison CSV and optional figure.


In [ ]:
#@title Runtime mode and base paths
from pathlib import Path
import os
import shutil
import sys

RUN_MODE = "local"  #@param ["local", "colab"]
if RUN_MODE not in {"local", "colab"}:
    raise ValueError("RUN_MODE must be either 'local' or 'colab'.")

# Local defaults are intended for Ubuntu + PyCharm/Jupyter from the repository root.
LOCAL_REPO_ROOT = "/home/mechti/PycharmProjects/DeepMzyme"  #@param {type:"string"}
LOCAL_DATA_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data"  #@param {type:"string"}
LOCAL_OUTPUT_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/notebook_outputs"  #@param {type:"string"}
LOCAL_BUNDLE_PATH = ""  #@param {type:"string"}
LOCAL_UNPACK_DIR = "/home/mechti/PycharmProjects/DeepMzyme/.notebook_bundle"  #@param {type:"string"}
LOCAL_EMBEDDINGS_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/embeddings"  #@param {type:"string"}
LOCAL_RING_EXE_PATH = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/ring-4.0/out/bin/ring"  #@param {type:"string"}
LOCAL_RING_EDGE_OUTPUT_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/embeddings"  #@param {type:"string"}
LOCAL_PRECOMPUTED_RING_EDGES_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/embeddings"  #@param {type:"string"}

# Colab defaults keep Drive for persistent data and /content for runtime-local work.
COLAB_REPO_ROOT = "/content/DeepMzyme"  #@param {type:"string"}
COLAB_DRIVE_DATA_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data"  #@param {type:"string"}
COLAB_OUTPUT_DIR = "/content/deepmzyme_outputs"  #@param {type:"string"}
COLAB_UNPACK_DIR = "/content/deepmzyme_bundle"  #@param {type:"string"}
COLAB_EMBEDDINGS_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/embeddings"  #@param {type:"string"}
COLAB_RING_EXE_PATH = "DeepMzyme_Data/ring-4.0/out/bin/ring"  #@param {type:"string"}
COLAB_RING_EDGE_OUTPUT_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/embeddings"  #@param {type:"string"}
COLAB_PRECOMPUTED_RING_EDGES_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/embeddings"  #@param {type:"string"}

REPO_GIT_URL = "https://github.com/MECHTI1/DeepMzyme.git"  #@param {type:"string"}
REPO_GIT_BRANCH = "main"  #@param {type:"string"}
DATASET_NAME = "train_and_test_sets_structures_non_overlapped_pinmymetal"  #@param {type:"string"}
BUNDLE_FILENAME = "train_and_test_sets_structures_non_overlapped_pinmymetal_colab_bundle_structures.tar.zst"  #@param {type:"string"}
DRIVE_OUTPUT_SUBDIR = "Train_Parameters_Models_and_Results"  #@param {type:"string"}

print("RUN_MODE:", RUN_MODE)
print("Current working directory:", Path.cwd())
print("Python executable:", sys.executable)
print("Expected local DeepMzyme interpreter:", "/home/mechti/miniconda3/envs/DeepMzyme/bin/python")
if RUN_MODE == "local" and Path(sys.executable).resolve() != Path("/home/mechti/miniconda3/envs/DeepMzyme/bin/python").resolve():
    print("Note: local mode uses the active notebook kernel. Switch the Jupyter/PyCharm kernel if you need the DeepMzyme conda environment.")



## 1. Optional Google Drive Mount

Drive is mounted only in `RUN_MODE == "colab"`. Local Ubuntu / PyCharm / Jupyter runs skip this section and use the local paths configured above.


In [ ]:
#@title Optional Google Drive mount
MOUNT_DRIVE = RUN_MODE == "colab"  #@param {type:"boolean"}

if RUN_MODE == "colab":
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Colab Drive mount skipped. Make sure bundle and output paths are accessible.')
elif RUN_MODE == "local":
    print('Local mode: Google Drive mount skipped.')



## 2. Resolve Repository, Data, Bundle, and Output Paths

This cell converts the runtime-mode settings into canonical path variables used by the rest of the notebook: `REPO_ROOT`, `DATA_DIR`, `BUNDLE_PATH`, `OUTPUT_DIR`, `EMBEDDINGS_DIR`, `RING_EXE_PATH`, `RING_EDGE_OUTPUT_DIR`, and `PRECOMPUTED_RING_EDGES_DIR`.

In local mode, the notebook assumes the repository already exists. In Colab mode, the dependency cell can clone the repository into `/content` when needed.


In [ ]:
#@title Resolve paths and run environment checks
from pathlib import Path
import os
import sys


def _path(value, *, resolve=False):
    path = Path(str(value)).expanduser()
    return path.resolve() if resolve else path

if RUN_MODE == "local":
    REPO_ROOT = _path(LOCAL_REPO_ROOT, resolve=True)
    DATA_DIR = _path(LOCAL_DATA_DIR, resolve=True)
    OUTPUT_DIR = _path(LOCAL_OUTPUT_DIR, resolve=True)
    UNPACK_DIR = _path(LOCAL_UNPACK_DIR, resolve=True)
    BUNDLE_PATH = _path(LOCAL_BUNDLE_PATH, resolve=True) if str(LOCAL_BUNDLE_PATH).strip() else DATA_DIR / 'DeepMzyme_Colab_Bundles' / BUNDLE_FILENAME
    EMBEDDINGS_DIR = _path(LOCAL_EMBEDDINGS_DIR, resolve=True)
    RING_EXE_PATH = _path(LOCAL_RING_EXE_PATH, resolve=True) if str(LOCAL_RING_EXE_PATH).strip() else Path('')
    RING_EDGE_OUTPUT_DIR = _path(LOCAL_RING_EDGE_OUTPUT_DIR, resolve=True)
    PRECOMPUTED_RING_EDGES_DIR = _path(LOCAL_PRECOMPUTED_RING_EDGES_DIR, resolve=True)
    DRIVE_OUTPUT_DIR = OUTPUT_DIR
elif RUN_MODE == "colab":
    REPO_ROOT = _path(COLAB_REPO_ROOT, resolve=True)
    DATA_DIR = _path(COLAB_DRIVE_DATA_DIR)
    OUTPUT_DIR = _path(COLAB_OUTPUT_DIR, resolve=True)
    UNPACK_DIR = _path(COLAB_UNPACK_DIR, resolve=True)
    BUNDLE_PATH = DATA_DIR / 'DeepMzyme_Colab_Bundles' / BUNDLE_FILENAME
    EMBEDDINGS_DIR = _path(COLAB_EMBEDDINGS_DIR)
    RING_EXE_PATH = _path(COLAB_RING_EXE_PATH)
    RING_EDGE_OUTPUT_DIR = _path(COLAB_RING_EDGE_OUTPUT_DIR)
    PRECOMPUTED_RING_EDGES_DIR = _path(COLAB_PRECOMPUTED_RING_EDGES_DIR)
    DRIVE_OUTPUT_DIR = DATA_DIR / DRIVE_OUTPUT_SUBDIR
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

# Backward-compatible aliases used by later notebook helpers.
repo_dir = REPO_ROOT
repo_root = REPO_ROOT
REPO_DIR = str(REPO_ROOT)
drive_data_dir = DATA_DIR
bundle_path = BUNDLE_PATH
unpack_dir = UNPACK_DIR
local_runs_dir = OUTPUT_DIR / 'runs'
drive_output_dir = DRIVE_OUTPUT_DIR

DEFAULT_ESM_EMBEDDINGS_DIR = str(EMBEDDINGS_DIR)
DEFAULT_RING_EXE_PATH = str(RING_EXE_PATH)
DEFAULT_RING_EDGE_OUTPUT_DIR = str(RING_EDGE_OUTPUT_DIR)
DEFAULT_PRECOMPUTED_RING_EDGES_DIR = str(PRECOMPUTED_RING_EDGES_DIR)

required_files = [
    REPO_ROOT / 'AGENTS.md',
    REPO_ROOT / 'Plan.md',
    REPO_ROOT / 'README.md',
    REPO_ROOT / 'src' / 'train.py',
    REPO_ROOT / 'src' / 'report_runs.py',
]

print('Current working directory:', Path.cwd())
print('Python executable:', sys.executable)
print('RUN_MODE:', RUN_MODE)
print('REPO_ROOT:', REPO_ROOT)
print('REPO_ROOT exists:', REPO_ROOT.exists())
print('DATA_DIR:', DATA_DIR)
print('DATA_DIR exists:', DATA_DIR.exists())
print('BUNDLE_PATH:', BUNDLE_PATH)
print('BUNDLE_PATH exists:', BUNDLE_PATH.exists())
print('UNPACK_DIR:', UNPACK_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('EMBEDDINGS_DIR:', EMBEDDINGS_DIR)
print('EMBEDDINGS_DIR exists:', EMBEDDINGS_DIR.exists())
print('RING_EXE_PATH:', RING_EXE_PATH if str(RING_EXE_PATH) else '(not configured)')
if str(RING_EXE_PATH):
    print('RING_EXE_PATH exists:', RING_EXE_PATH.exists())
    print('RING_EXE_PATH executable:', os.access(RING_EXE_PATH, os.X_OK) if RING_EXE_PATH.exists() else False)
print('RING_EDGE_OUTPUT_DIR:', RING_EDGE_OUTPUT_DIR)
print('PRECOMPUTED_RING_EDGES_DIR:', PRECOMPUTED_RING_EDGES_DIR)
print('Required files:')
for path in required_files:
    print(' ', path, 'exists:', path.exists())



## 3. Repository and Dependency Checks

Colab mode can install dependencies in a fresh runtime. Local mode does not pip-install automatically; use the active Jupyter/PyCharm kernel from the DeepMzyme conda environment, or explicitly set `INSTALL_REQUIREMENTS=True` if you want this notebook to install into that kernel.


In [ ]:
#@title Repository and dependency checks
import os
import subprocess
import sys
from pathlib import Path

INSTALL_REQUIREMENTS = RUN_MODE == "colab"  #@param {type:"boolean"}
CHECK_IMPORTS = True  #@param {type:"boolean"}

if RUN_MODE == "colab":
    if not REPO_GIT_URL or '<' in REPO_GIT_URL:
        raise ValueError('Set REPO_GIT_URL to your DeepMzyme repository URL before cloning.')
    if repo_dir.exists():
        print(f'Repository already exists at {repo_dir}; skipping clone.')
        print('Warning: an existing checkout may be stale or may contain local work. This notebook will not overwrite it automatically.')
        for git_cmd in (['git', 'branch', '--show-current'], ['git', 'rev-parse', 'HEAD'], ['git', 'status', '--short']):
            label = ' '.join(git_cmd)
            result = subprocess.run(git_cmd, cwd=repo_dir, check=False, capture_output=True, text=True)
            output = result.stdout.strip() or result.stderr.strip() or '(no output)'
            print(f'$ {label}\\n{output}')
    else:
        subprocess.run(['git', 'clone', '--branch', REPO_GIT_BRANCH, REPO_GIT_URL, str(repo_dir)], check=True)
elif RUN_MODE == "local":
    if not repo_dir.exists():
        raise FileNotFoundError(f'Local REPO_ROOT does not exist: {repo_dir}')
    print(f'Local repository: {repo_dir}')
    for git_cmd in (['git', 'branch', '--show-current'], ['git', 'rev-parse', 'HEAD'], ['git', 'status', '--short']):
        label = ' '.join(git_cmd)
        result = subprocess.run(git_cmd, cwd=repo_dir, check=False, capture_output=True, text=True)
        output = result.stdout.strip() or result.stderr.strip() or '(no output)'
        print(f'$ {label}\\n{output}')

requirements_path = repo_dir / 'src' / 'requirements.txt'
if INSTALL_REQUIREMENTS:
    if requirements_path.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
    else:
        if RUN_MODE != "colab":
            raise FileNotFoundError(f'Requirements file not found: {requirements_path}')
        subprocess.run([
            sys.executable, '-m', 'pip', 'install',
            'torch-geometric==2.7.0', 'biopython==1.87', 'biotite>=1.6', 'propka>=3.5', 'gemmi>=0.7', 'pandas', 'matplotlib'
        ], check=True)
else:
    print('Dependency installation skipped. Set INSTALL_REQUIREMENTS=True to install into the active notebook kernel.')

if CHECK_IMPORTS:
    src_path = str(repo_dir / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    import torch
    import pandas as pd
    import torch_geometric
    print('Python:', sys.executable)
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('PyTorch Geometric:', torch_geometric.__version__)



## 4. Optional Bundle Unpack

Colab mode unpacks the compressed bundle by default. Local mode skips unpacking by default and searches `DATA_DIR`, `UNPACK_DIR`, and `REPO_ROOT` for the dataset; enable `UNPACK_BUNDLE` if you want to extract a local bundle.


In [ ]:
#@title Optional bundle unpack
import shutil
import subprocess
from pathlib import Path

UNPACK_BUNDLE = RUN_MODE == "colab"  #@param {type:"boolean"}
FORCE_REUNPACK = False  #@param {type:"boolean"}
INSTALL_ZSTD_IN_COLAB = True  #@param {type:"boolean"}


def run_checked(cmd):
    print('+', ' '.join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)


def unpack_archive(source: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(source.suffixes).lower()
    if suffixes.endswith('.tar.zst'):
        if shutil.which('zstd') is None:
            if RUN_MODE == "colab" and INSTALL_ZSTD_IN_COLAB:
                run_checked(['apt-get', 'update'])
                run_checked(['apt-get', 'install', '-y', 'zstd'])
            else:
                raise RuntimeError('zstd is required to unpack .tar.zst bundles. Install zstd or set UNPACK_BUNDLE=False.')
        run_checked(['tar', '--use-compress-program=zstd', '-xf', source, '-C', destination])
    elif suffixes.endswith(('.tar.gz', '.tgz', '.tar')):
        run_checked(['tar', '-xf', source, '-C', destination])
    else:
        raise ValueError(f'Unsupported bundle archive type: {source}')

if UNPACK_BUNDLE:
    if FORCE_REUNPACK and unpack_dir.exists():
        shutil.rmtree(unpack_dir)

    if bundle_path.is_dir():
        unpack_dir = bundle_path.resolve()
        UNPACK_DIR = unpack_dir
        print(f'Using already-unpacked bundle directory: {unpack_dir}')
    else:
        if not bundle_path.exists():
            raise FileNotFoundError(f'Bundle not found: {bundle_path}')
        marker = unpack_dir / '.deepmzyme_bundle_unpacked'
        if marker.exists() and not FORCE_REUNPACK:
            print(f'Bundle already unpacked at {unpack_dir}. Set FORCE_REUNPACK=True to extract again.')
        else:
            unpack_archive(bundle_path, unpack_dir)
            marker.write_text('ok\\n', encoding='utf-8')
            print(f'Unpacked bundle into {unpack_dir}')
else:
    print('Bundle unpack skipped. Dataset discovery will search UNPACK_DIR, DATA_DIR, and REPO_ROOT.')



## 5. Verify Train/Test Directories and Summary CSVs

DeepMzyme training needs a train structure directory, a test structure directory, and MAHOMES-style summary CSVs with site-level columns such as PDB ID, EC number, metal residue number, and metal residue type. The bundle may also contain structure-level CSV artifacts; those are useful metadata, but they are not used here unless they match the training loader requirements.

In [ ]:
#@title Detect and verify dataset paths
import csv
from pathlib import Path

TRAIN_DIR_OVERRIDE = ''  #@param {type:"string"}
TEST_DIR_OVERRIDE = ''  #@param {type:"string"}
TRAIN_CSV_OVERRIDE = ''  #@param {type:"string"}
TEST_CSV_OVERRIDE = ''  #@param {type:"string"}

REQUIRED_SUMMARY_ALIASES = {
    'pdbid': {'pdbid', 'structure'},
    'metal residue number': {'metal residue number', 'chain_resi'},
    'EC number': {'ec number', 'ecnumber'},
    'metal residue type': {'metal residue type', 'metaltype'},
}


def has_required_summary_columns(path: Path) -> bool:
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            fields = {field.strip().lower() for field in (reader.fieldnames or []) if field}
    except Exception:
        return False
    return all(fields.intersection(aliases) for aliases in REQUIRED_SUMMARY_ALIASES.values())


def validate_site_level_summary_csv(path: Path, split_name: str) -> None:
    if has_required_summary_columns(path):
        return
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            fields = [field.strip() for field in (reader.fieldnames or []) if field]
    except Exception as exc:
        raise ValueError(f'Could not read {split_name} summary CSV {path}: {exc}') from exc
    raise ValueError(
        f'{split_name} summary CSV is not a site-level MAHOMES training summary: {path}. '
        f'Observed columns: {fields}. Structure-level metadata such as '
        'structure_name/ec_numbers/metal_type is not sufficient for training; rebuild the Colab bundle with the site-level summary CSVs.'
    )


def find_dataset_root(root: Path, dataset_name: str) -> Path:
    root = Path(root).expanduser()
    candidates = [
        root / 'DeepMzyme_Data' / dataset_name,
        root / dataset_name,
        root,
    ]
    if root.exists():
        candidates.extend(path for path in root.rglob(dataset_name) if path.is_dir())
    for candidate in candidates:
        if (candidate / 'train').is_dir() and (candidate / 'test').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find train/test directories for dataset {dataset_name!r} under {root}')


def find_dataset_root_in_roots(roots, dataset_name: str) -> Path:
    errors = []
    for root in roots:
        try:
            return find_dataset_root(Path(root), dataset_name)
        except FileNotFoundError as exc:
            errors.append(str(exc))
    raise FileNotFoundError('Could not find dataset root. Checked:\\n- ' + '\\n- '.join(errors))


def find_summary_csv(split_dir: Path, split_name: str) -> Path:
    preferred_names = [
        'catalytic_only_summary.csv',
        'final_data_summarazing_table_transition_metals_only_catalytic.csv',
        f'{DATASET_NAME}_{split_name}.csv',
    ]
    candidates = [split_dir / name for name in preferred_names]
    candidates.extend(sorted(split_dir.glob('*.csv')))
    for root in [unpack_dir, DATA_DIR, REPO_ROOT]:
        root = Path(root)
        if root.exists():
            candidates.extend(sorted(root.glob(f'**/*{split_name}*.csv')))
    seen = set()
    unique_candidates = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)
    for candidate in unique_candidates:
        if candidate.exists() and has_required_summary_columns(candidate):
            return candidate.resolve()
    found = [str(path) for path in unique_candidates if path.exists()]
    raise FileNotFoundError(
        f'No MAHOMES-style {split_name} summary CSV found. Checked: {found}. '
        'If your bundle only contains structure_name/ec_numbers/metal_type CSVs, rebuild it with the site-level summary CSV included.'
    )

search_roots = [unpack_dir, DATA_DIR, REPO_ROOT]
dataset_root = find_dataset_root_in_roots(search_roots, DATASET_NAME)
train_dir = Path(TRAIN_DIR_OVERRIDE).expanduser().resolve() if TRAIN_DIR_OVERRIDE else dataset_root / 'train'
test_dir = Path(TEST_DIR_OVERRIDE).expanduser().resolve() if TEST_DIR_OVERRIDE else dataset_root / 'test'
train_csv = Path(TRAIN_CSV_OVERRIDE).expanduser().resolve() if TRAIN_CSV_OVERRIDE else find_summary_csv(train_dir, 'train')
test_csv = Path(TEST_CSV_OVERRIDE).expanduser().resolve() if TEST_CSV_OVERRIDE else find_summary_csv(test_dir, 'test')

for label, path in [('train_dir', train_dir), ('test_dir', test_dir), ('train_csv', train_csv), ('test_csv', test_csv)]:
    if not path.exists():
        raise FileNotFoundError(f'{label} does not exist: {path}')
validate_site_level_summary_csv(train_csv, 'train')
validate_site_level_summary_csv(test_csv, 'test')

train_structures = sorted(list(train_dir.glob('*.pdb')) + list(train_dir.glob('*.cif')) + list(train_dir.glob('*.mmcif')))
test_structures = sorted(list(test_dir.glob('*.pdb')) + list(test_dir.glob('*.cif')) + list(test_dir.glob('*.mmcif')))
if not train_structures or not test_structures:
    raise ValueError(f'Expected structure files in both train and test directories: {train_dir}, {test_dir}')

print('Dataset search roots:', [str(root) for root in search_roots])
print('Dataset root:', dataset_root)
print('Train structures:', len(train_structures), train_dir)
print('Test structures:', len(test_structures), test_dir)
print('Train summary CSV:', train_csv)
print('Test summary CSV:', test_csv)



## 6. Set Default Run Values

These values provide defaults for the experiment grid and keep the notebook runnable without the optional widget panel. The widget panel below is the preferred interface: one selected value per grid field gives one run; multiple selected values or CSV entries create a sweep.

`RING edges mode` controls extra RING interaction edges on top of the normal radius graph:

- `Radius only (no RING interaction edges)`: do not pass `--use-ring-edges`; this is the safest baseline.
- `Radius + existing RING edge files`: pass precomputed RING edge files from `PRECOMPUTED_RING_EDGES_DIR`.
- `Radius + generate missing RING edge files`: ask training/preflight logic to generate missing RING edge files into `RING_EDGE_OUTPUT_DIR`.


In [ ]:
#@title Default run values
from datetime import datetime
from pathlib import Path

RING_EDGE_MODE_LABEL_TO_VALUE = {
    'Radius only (no RING interaction edges)': 'radius_only',
    'Radius + existing RING edge files': 'radius_plus_precomputed_ring',
    'Radius + generate missing RING edge files': 'generate_missing_ring',
}
RING_EDGE_MODE_VALUE_TO_LABEL = {value: label for label, value in RING_EDGE_MODE_LABEL_TO_VALUE.items()}


def normalize_ring_edges_mode(value):
    text = str(value).strip()
    if text in RING_EDGE_MODE_LABEL_TO_VALUE:
        return RING_EDGE_MODE_LABEL_TO_VALUE[text]
    if text in RING_EDGE_MODE_VALUE_TO_LABEL:
        return text
    valid = ', '.join(RING_EDGE_MODE_LABEL_TO_VALUE)
    raise ValueError(f'Unsupported RING edge mode {value!r}. Choose one of: {valid}')


def ring_edges_mode_label(value):
    mode = normalize_ring_edges_mode(value)
    return RING_EDGE_MODE_VALUE_TO_LABEL[mode]

TASK_MODE = 'metal_6_class'  #@param ['metal_6_class', 'metal_collapsed4_metric', 'ec_prediction']
MODEL_PRESET = 'Only-GVP'  #@param ['Only-GVP', 'Only-ESM', 'GVP + late fusion', 'GVP + early fusion', 'GVP + cross-modal attention']

EPOCHS = 50  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
LEARNING_RATE = 3e-5  #@param {type:"number"}
WEIGHT_DECAY = 1e-4  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
VAL_FRACTION = 0.15  #@param {type:"number"}
DEVICE = 'cuda'  #@param ['cuda', 'cpu']
RUN_NAME = ''  #@param {type:"string"}

EC_LABEL_DEPTH = 1  #@param {type:"integer"}
EC_CONTRASTIVE_WEIGHT = 0.0  #@param {type:"number"}
NODE_FEATURE_SET = 'conservative'  #@param ['conservative']
SPLIT_BY = 'pdbid'  #@param ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id']
LR_SCHEDULE = 'fixed'  #@param ['fixed', 'cosine', 'step']
LR_STEP_SIZE = 10  #@param {type:"integer"}
LR_DECAY_GAMMA = 0.5  #@param {type:"number"}

CROSS_ATTENTION_LAYERS = 1  #@param {type:"integer"}
CROSS_ATTENTION_HEADS = 4  #@param {type:"integer"}
CROSS_ATTENTION_DROPOUT = 0.1  #@param {type:"number"}
CROSS_ATTENTION_NEIGHBORHOOD = 'all'  #@param ['all', 'first_shell', 'first_second_shell']
CROSS_ATTENTION_BIDIRECTIONAL = False  #@param {type:"boolean"}
SINGLE_CROSS_ATTENTION_LAYERS = CROSS_ATTENTION_LAYERS
SINGLE_CROSS_ATTENTION_HEADS = CROSS_ATTENTION_HEADS

USE_ESM_EMBEDDINGS = True  #@param {type:"boolean"}
ESM_EMBEDDINGS_DIR = DEFAULT_ESM_EMBEDDINGS_DIR  #@param {type:"string"}
PREPARE_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_EXTERNAL_FEATURES = True  #@param {type:"boolean"}

RING_EDGES_MODE = 'Radius only (no RING interaction edges)'  #@param ['Radius only (no RING interaction edges)', 'Radius + existing RING edge files', 'Radius + generate missing RING edge files']
RING_EXE_PATH = DEFAULT_RING_EXE_PATH  #@param {type:"string"}
RING_EDGE_OUTPUT_DIR = DEFAULT_RING_EDGE_OUTPUT_DIR  #@param {type:"string"}
PRECOMPUTED_RING_EDGES_DIR = DEFAULT_PRECOMPUTED_RING_EDGES_DIR  #@param {type:"string"}
REQUIRE_RING_EDGES = False  #@param {type:"boolean"}
PREPARE_MISSING_RING_EDGES = False  #@param {type:"boolean"}
RING_EDGES_MODE = normalize_ring_edges_mode(RING_EDGES_MODE)

RUN_HELD_OUT_TEST_EVAL = True  #@param {type:"boolean"}

task_map = {
    'metal_6_class': ('metal', 'val_metal_balanced_acc'),
    'metal_collapsed4_metric': ('metal', 'val_metal_collapsed4_balanced_acc'),
    'ec_prediction': ('ec', 'val_ec_group_balanced_acc'),
}
preset_map = {
    'Only-GVP': {'model_architecture': 'only_gvp', 'uses_esm': False},
    'Only-ESM': {'model_architecture': 'only_esm', 'uses_esm': True},
    'GVP + late fusion': {'model_architecture': 'gvp', 'fusion_mode': 'late_fusion', 'uses_esm': True},
    'GVP + early fusion': {'model_architecture': 'gvp', 'fusion_mode': 'early_fusion', 'early_esm_dim': 32, 'early_esm_dropout': 0.2, 'uses_esm': True},
    'GVP + cross-modal attention': {'model_architecture': 'gvp', 'fusion_mode': 'cross_modal_attention', 'uses_esm': True},
}

task_name, selection_metric = task_map[TASK_MODE]
preset = preset_map[MODEL_PRESET]
print('Task:', task_name)
print('Selection metric:', selection_metric)
print('Model preset:', MODEL_PRESET)
print('Selected model uses ESM:', bool(preset.get('uses_esm')))
print('ESM embeddings enabled for ESM presets:', USE_ESM_EMBEDDINGS)
print('RING edge mode:', ring_edges_mode_label(RING_EDGES_MODE), f'[{RING_EDGES_MODE}]')
print('Requested run name:', RUN_NAME or '(auto)')


## 7. Configure Multi-Run Sweep Mode

Sweep mode expands the selected lists into planned experiments with `itertools.product`, then runs each plan through `src/train.py`. Use validation metrics first when comparing sweep results; held-out test metrics are final reporting for the validation-selected checkpoint only and should not be used to choose hyperparameters.

In [ ]:
#@title Sweep configuration
# Keep these lists short until the full workflow is verified in Colab.
TASK_MODES = ['metal_6_class']
MODEL_PRESETS = ['Only-GVP']
# Add more RING edge labels here for a sweep, for example: 'Radius + existing RING edge files'.
RING_EDGES_MODES = ['Radius only (no RING interaction edges)']
LEARNING_RATES = [3e-5]
WEIGHT_DECAYS = [1e-4]
BATCH_SIZES = [8]
SEEDS = [42]
NODE_FEATURE_SETS = ['conservative']
EC_LABEL_DEPTHS = [1]
EC_CONTRASTIVE_WEIGHTS = [0.0]
CROSS_ATTENTION_LAYERS = [1]
CROSS_ATTENTION_HEADS = [4]
LR_SCHEDULES = ['fixed']

MAX_SWEEP_RUNS = 24  #@param {type:"integer"}
STOP_ON_FIRST_FAILURE = True  #@param {type:"boolean"}
SKIP_EXISTING_RUNS = True  #@param {type:"boolean"}

print('Sweep task modes:', TASK_MODES)
print('Sweep model presets:', MODEL_PRESETS)
print('Available RING edge modes:', list(RING_EDGE_MODE_LABEL_TO_VALUE))
print('Sweep RING edge modes:', [ring_edges_mode_label(mode) for mode in RING_EDGES_MODES])
print('Maximum planned runs:', MAX_SWEEP_RUNS)



## Optional Interactive Configuration Panel

The widget panel below is optional. One experiment grid controls both one-off runs and sweeps: select one value per field for one run, or multiple values for a sweep.

Continuous numeric lists such as learning rates, weight decays, batch sizes, seeds, and EC contrastive weights use comma-separated text. Finite option lists such as task mode, model preset, EC label depth, cross-attention layers/heads, and LR schedule use multi-select widgets. If you do not run the widget cells, the existing `#@param` configuration cells above still define the final settings used by the command builder.


In [ ]:
#@title Interactive configuration panel
from IPython.display import display

try:
    import ipywidgets as widgets
    _WIDGET_PANEL_AVAILABLE = True
except ModuleNotFoundError:
    _WIDGET_PANEL_AVAILABLE = False

    class _FallbackControl:
        def __init__(self, value=None, children=None):
            self.value = value
            self.children = children or []

        def set_title(self, index, title):
            return None

    class _FallbackWidgets:
        def Layout(self, **kwargs):
            return None

        def Dropdown(self, *, value='', **kwargs):
            return _FallbackControl(value=value)

        def IntText(self, *, value=0, **kwargs):
            return _FallbackControl(value=value)

        def FloatText(self, *, value=0.0, **kwargs):
            return _FallbackControl(value=value)

        def Text(self, *, value='', **kwargs):
            return _FallbackControl(value=value)

        def Checkbox(self, *, value=False, **kwargs):
            return _FallbackControl(value=value)

        def SelectMultiple(self, *, value=(), **kwargs):
            return _FallbackControl(value=tuple(value))

        def HBox(self, children):
            return _FallbackControl(children=list(children))

        def VBox(self, children):
            return _FallbackControl(children=list(children))

        def Accordion(self, children):
            return _FallbackControl(children=list(children))

    widgets = _FallbackWidgets()

_WIDGET_STYLE = {'description_width': '190px'}
_WIDGET_LAYOUT = widgets.Layout(width='520px')
_TEXT_LAYOUT = widgets.Layout(width='520px')
_MULTI_LAYOUT = widgets.Layout(width='520px')


def _current_value(name, default):
    return globals().get(name, default)


def _as_list(value):
    if isinstance(value, (list, tuple, set)):
        return list(value)
    return [value]


def _selected_defaults(options, defaults):
    selected = [value for value in _as_list(defaults) if value in options]
    return tuple(selected or options[:1])


def _choice(description, options, value):
    options = list(options)
    value = str(value).strip()
    if value not in options:
        value = options[0]
    return widgets.Dropdown(description=description, options=options, value=value, style=_WIDGET_STYLE, layout=_WIDGET_LAYOUT)


def _multi_choice(description, options, selected_values):
    options = list(options)
    return widgets.SelectMultiple(
        description=description,
        options=options,
        value=_selected_defaults(options, selected_values),
        rows=min(max(len(options), 3), 7),
        style=_WIDGET_STYLE,
        layout=_MULTI_LAYOUT,
    )


def _int_text(description, value):
    return widgets.IntText(description=description, value=int(value), style=_WIDGET_STYLE, layout=_WIDGET_LAYOUT)


def _float_text(description, value):
    return widgets.FloatText(description=description, value=float(value), style=_WIDGET_STYLE, layout=_WIDGET_LAYOUT)


def _text(description, value='', placeholder=''):
    return widgets.Text(description=description, value=str(value), placeholder=placeholder, style=_WIDGET_STYLE, layout=_TEXT_LAYOUT)


def _csv_text(description, values, placeholder):
    text = ','.join(str(value) for value in _as_list(values))
    return _text(description, text, placeholder)


def _checkbox(description, value):
    return widgets.Checkbox(description=description, value=bool(value), indent=False, layout=_WIDGET_LAYOUT)


_run_widgets = {
    'EPOCHS': _int_text('Epochs', _current_value('EPOCHS', 50)),
    'VAL_FRACTION': _float_text('Validation fraction', _current_value('VAL_FRACTION', 0.15)),
    'DEVICE': _choice('Device', ['cuda', 'cpu'], _current_value('DEVICE', 'cuda')),
    'RUN_NAME': _text('Run name', _current_value('RUN_NAME', ''), 'optional; only used when grid has one run'),
    'SPLIT_BY': _choice('Split by', ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id'], _current_value('SPLIT_BY', 'pdbid')),
    'LR_STEP_SIZE': _int_text('LR step size', _current_value('LR_STEP_SIZE', 10)),
    'LR_DECAY_GAMMA': _float_text('LR decay gamma', _current_value('LR_DECAY_GAMMA', 0.5)),
    'CROSS_ATTENTION_DROPOUT': _float_text('Cross-attn dropout', _current_value('CROSS_ATTENTION_DROPOUT', 0.1)),
    'CROSS_ATTENTION_NEIGHBORHOOD': _choice('Cross-attn neighborhood', ['all', 'first_shell', 'first_second_shell'], _current_value('CROSS_ATTENTION_NEIGHBORHOOD', 'all')),
    'CROSS_ATTENTION_BIDIRECTIONAL': _checkbox('Cross-attn bidirectional', _current_value('CROSS_ATTENTION_BIDIRECTIONAL', False)),
    'RUN_HELD_OUT_TEST_EVAL': _checkbox('Run held-out test eval', _current_value('RUN_HELD_OUT_TEST_EVAL', True)),
}

_esm_widgets = {
    'USE_ESM_EMBEDDINGS': _checkbox('Use ESM embeddings', _current_value('USE_ESM_EMBEDDINGS', True)),
    'ESM_EMBEDDINGS_DIR': _text('ESM embeddings dir', _current_value('ESM_EMBEDDINGS_DIR', DEFAULT_ESM_EMBEDDINGS_DIR)),
    'PREPARE_MISSING_ESM_EMBEDDINGS': _checkbox('Prepare missing ESM embeddings', _current_value('PREPARE_MISSING_ESM_EMBEDDINGS', False)),
    'ALLOW_MISSING_ESM_EMBEDDINGS': _checkbox('Allow missing ESM embeddings', _current_value('ALLOW_MISSING_ESM_EMBEDDINGS', False)),
    'ALLOW_MISSING_EXTERNAL_FEATURES': _checkbox('Allow missing external features', _current_value('ALLOW_MISSING_EXTERNAL_FEATURES', True)),
}

_ring_widgets = {
    'RING_EXE_PATH': _text('RING executable', _current_value('RING_EXE_PATH', DEFAULT_RING_EXE_PATH)),
    'RING_EDGE_OUTPUT_DIR': _text('RING output dir', _current_value('RING_EDGE_OUTPUT_DIR', DEFAULT_RING_EDGE_OUTPUT_DIR)),
    'PRECOMPUTED_RING_EDGES_DIR': _text('Precomputed RING dir', _current_value('PRECOMPUTED_RING_EDGES_DIR', DEFAULT_PRECOMPUTED_RING_EDGES_DIR)),
    'REQUIRE_RING_EDGES': _checkbox('Require RING edges', _current_value('REQUIRE_RING_EDGES', False)),
    'PREPARE_MISSING_RING_EDGES': _checkbox('Prepare missing RING edges', _current_value('PREPARE_MISSING_RING_EDGES', False)),
}

_grid_select_widgets = {
    'TASK_MODES': _multi_choice('Task modes', ['metal_6_class', 'metal_collapsed4_metric', 'ec_prediction'], _current_value('TASK_MODES', ['metal_6_class'])),
    'MODEL_PRESETS': _multi_choice('Model presets', ['Only-GVP', 'Only-ESM', 'GVP + late fusion', 'GVP + early fusion', 'GVP + cross-modal attention'], _current_value('MODEL_PRESETS', ['Only-GVP'])),
    'RING_EDGES_MODES': _multi_choice('RING modes', list(RING_EDGE_MODE_LABEL_TO_VALUE), [ring_edges_mode_label(mode) for mode in _current_value('RING_EDGES_MODES', ['radius_only'])]),
    'NODE_FEATURE_SETS': _multi_choice('Node feature sets', ['conservative'], _current_value('NODE_FEATURE_SETS', ['conservative'])),
    'EC_LABEL_DEPTHS': _multi_choice('EC label depths', [1, 2, 3], _current_value('EC_LABEL_DEPTHS', [1])),
    'CROSS_ATTENTION_LAYERS': _multi_choice('Cross-attn layers', [1, 2, 3], _current_value('CROSS_ATTENTION_LAYERS', [1])),
    'CROSS_ATTENTION_HEADS': _multi_choice('Cross-attn heads', [2, 4, 8], _current_value('CROSS_ATTENTION_HEADS', [4])),
    'LR_SCHEDULES': _multi_choice('LR schedules', ['fixed', 'cosine', 'step'], _current_value('LR_SCHEDULES', ['fixed'])),
}

_grid_csv_widgets = {
    'LEARNING_RATES': _csv_text('Learning rates CSV', _current_value('LEARNING_RATES', [3e-5]), '3e-5,1e-4'),
    'WEIGHT_DECAYS': _csv_text('Weight decays CSV', _current_value('WEIGHT_DECAYS', [1e-4]), '1e-4,0'),
    'BATCH_SIZES': _csv_text('Batch sizes CSV', _current_value('BATCH_SIZES', [8]), '4,8,16'),
    'SEEDS': _csv_text('Seeds CSV', _current_value('SEEDS', [42]), '42,7,123'),
    'EC_CONTRASTIVE_WEIGHTS': _csv_text('EC contrast CSV', _current_value('EC_CONTRASTIVE_WEIGHTS', [0.0]), '0,0.1,0.5'),
}

_sweep_controls = {
    'MAX_SWEEP_RUNS': _int_text('Max grid runs', _current_value('MAX_SWEEP_RUNS', 24)),
    'STOP_ON_FIRST_FAILURE': _checkbox('Stop on first failure', _current_value('STOP_ON_FIRST_FAILURE', True)),
    'SKIP_EXISTING_RUNS': _checkbox('Skip existing runs', _current_value('SKIP_EXISTING_RUNS', True)),
}

_execution_widgets = {
    'EXECUTION_MODE': _choice('Execution mode', ['preview_only', 'single_run', 'sweep', 'auto'], _current_value('EXECUTION_MODE', 'preview_only')),
    'COPY_OUTPUTS_TO_DRIVE': _checkbox('Copy outputs to Drive', _current_value('COPY_OUTPUTS_TO_DRIVE', RUN_MODE == 'colab')),
}

_panel = widgets.Accordion(children=[
    widgets.VBox(list(_run_widgets.values())),
    widgets.VBox(list(_esm_widgets.values())),
    widgets.VBox(list(_ring_widgets.values())),
    widgets.VBox(list(_grid_select_widgets.values()) + list(_grid_csv_widgets.values()) + list(_sweep_controls.values())),
    widgets.VBox(list(_execution_widgets.values())),
])
for index, title in enumerate(['Shared run settings', 'ESM / external features', 'RING edge paths', 'Experiment grid', 'Execution controls']):
    _panel.set_title(index, title)

if _WIDGET_PANEL_AVAILABLE:
    print('Optional grid panel displayed. Select one value per field for one run, or multiple values for a sweep. Run the next cell to read these values into the final notebook configuration.')
    display(_panel)
else:
    print('ipywidgets is not installed; interactive panel skipped.')
    print('Existing #@param/default values were captured, so the next configuration-read cell can still run.')


In [ ]:
#@title Read widget values into final configuration variables
import json

if '_run_widgets' not in globals() or '_grid_select_widgets' not in globals() or '_grid_csv_widgets' not in globals() or '_execution_widgets' not in globals():
    raise RuntimeError("Run the 'Interactive configuration panel' cell first, or skip both widget cells and use the existing #@param configuration cells.")


def _csv_tokens(text):
    return [token.strip() for token in str(text or '').split(',') if token.strip()]


def _parse_csv_numbers(text, converter, label):
    tokens = _csv_tokens(text)
    if not tokens:
        raise ValueError(f'{label} must contain at least one comma-separated value.')
    parsed = []
    for value in tokens:
        try:
            parsed.append(converter(value))
        except (TypeError, ValueError) as exc:
            raise ValueError(f'Could not parse {label} value {value!r} from comma-separated values: {text!r}') from exc
    return parsed


def parse_csv_ints(text, label):
    return _parse_csv_numbers(text, int, label)


def parse_csv_floats(text, label):
    return _parse_csv_numbers(text, float, label)


def _text_value(widgets_by_name, name):
    return str(widgets_by_name[name].value).strip()


def _choice_value(widgets_by_name, name):
    return str(widgets_by_name[name].value).strip()


def _selected_values(name):
    return list(_grid_select_widgets[name].value)


def _validate_choice(name, value, options):
    if value not in options:
        valid = ', '.join(str(option) for option in options)
        raise ValueError(f'{name}={value!r} is not valid. Choose one of: {valid}')


def _validate_choices(name, values, options):
    invalid = [value for value in values if value not in options]
    if invalid:
        valid = ', '.join(str(option) for option in options)
        raise ValueError(f'{name} contains invalid value(s) {invalid!r}. Choose from: {valid}')
    if not values:
        raise ValueError(f'{name} must contain at least one value.')


TASK_MODE_OPTIONS = list(task_map)
MODEL_PRESET_OPTIONS = list(preset_map)
DEVICE_OPTIONS = ['cuda', 'cpu']
NODE_FEATURE_SET_OPTIONS = ['conservative']
SPLIT_BY_OPTIONS = ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id']
LR_SCHEDULE_OPTIONS = ['fixed', 'cosine', 'step']
CROSS_ATTENTION_NEIGHBORHOOD_OPTIONS = ['all', 'first_shell', 'first_second_shell']
EXECUTION_MODE_OPTIONS = ['preview_only', 'single_run', 'sweep', 'auto']
EC_LABEL_DEPTH_OPTIONS = [1, 2, 3]
CROSS_ATTENTION_LAYER_OPTIONS = [1, 2, 3]
CROSS_ATTENTION_HEAD_OPTIONS = [2, 4, 8]

EPOCHS = int(_run_widgets['EPOCHS'].value)
VAL_FRACTION = float(_run_widgets['VAL_FRACTION'].value)
DEVICE = _choice_value(_run_widgets, 'DEVICE')
RUN_NAME = _text_value(_run_widgets, 'RUN_NAME')
SPLIT_BY = _choice_value(_run_widgets, 'SPLIT_BY')
LR_STEP_SIZE = int(_run_widgets['LR_STEP_SIZE'].value)
LR_DECAY_GAMMA = float(_run_widgets['LR_DECAY_GAMMA'].value)
CROSS_ATTENTION_DROPOUT = float(_run_widgets['CROSS_ATTENTION_DROPOUT'].value)
CROSS_ATTENTION_NEIGHBORHOOD = _choice_value(_run_widgets, 'CROSS_ATTENTION_NEIGHBORHOOD')
CROSS_ATTENTION_BIDIRECTIONAL = bool(_run_widgets['CROSS_ATTENTION_BIDIRECTIONAL'].value)
RUN_HELD_OUT_TEST_EVAL = bool(_run_widgets['RUN_HELD_OUT_TEST_EVAL'].value)

USE_ESM_EMBEDDINGS = bool(_esm_widgets['USE_ESM_EMBEDDINGS'].value)
ESM_EMBEDDINGS_DIR = _text_value(_esm_widgets, 'ESM_EMBEDDINGS_DIR')
PREPARE_MISSING_ESM_EMBEDDINGS = bool(_esm_widgets['PREPARE_MISSING_ESM_EMBEDDINGS'].value)
ALLOW_MISSING_ESM_EMBEDDINGS = bool(_esm_widgets['ALLOW_MISSING_ESM_EMBEDDINGS'].value)
ALLOW_MISSING_EXTERNAL_FEATURES = bool(_esm_widgets['ALLOW_MISSING_EXTERNAL_FEATURES'].value)

RING_EXE_PATH = _text_value(_ring_widgets, 'RING_EXE_PATH')
RING_EDGE_OUTPUT_DIR = _text_value(_ring_widgets, 'RING_EDGE_OUTPUT_DIR')
PRECOMPUTED_RING_EDGES_DIR = _text_value(_ring_widgets, 'PRECOMPUTED_RING_EDGES_DIR')
REQUIRE_RING_EDGES = bool(_ring_widgets['REQUIRE_RING_EDGES'].value)
PREPARE_MISSING_RING_EDGES = bool(_ring_widgets['PREPARE_MISSING_RING_EDGES'].value)

TASK_MODES = _selected_values('TASK_MODES')
MODEL_PRESETS = _selected_values('MODEL_PRESETS')
RING_EDGES_MODES = [normalize_ring_edges_mode(mode) for mode in _selected_values('RING_EDGES_MODES')]
NODE_FEATURE_SETS = _selected_values('NODE_FEATURE_SETS')
EC_LABEL_DEPTHS = [int(value) for value in _selected_values('EC_LABEL_DEPTHS')]
CROSS_ATTENTION_LAYERS = [int(value) for value in _selected_values('CROSS_ATTENTION_LAYERS')]
CROSS_ATTENTION_HEADS = [int(value) for value in _selected_values('CROSS_ATTENTION_HEADS')]
LR_SCHEDULES = _selected_values('LR_SCHEDULES')

LEARNING_RATES = parse_csv_floats(_grid_csv_widgets['LEARNING_RATES'].value, 'LEARNING_RATES')
WEIGHT_DECAYS = parse_csv_floats(_grid_csv_widgets['WEIGHT_DECAYS'].value, 'WEIGHT_DECAYS')
BATCH_SIZES = parse_csv_ints(_grid_csv_widgets['BATCH_SIZES'].value, 'BATCH_SIZES')
SEEDS = parse_csv_ints(_grid_csv_widgets['SEEDS'].value, 'SEEDS')
EC_CONTRASTIVE_WEIGHTS = parse_csv_floats(_grid_csv_widgets['EC_CONTRASTIVE_WEIGHTS'].value, 'EC_CONTRASTIVE_WEIGHTS')

MAX_SWEEP_RUNS = int(_sweep_controls['MAX_SWEEP_RUNS'].value)
STOP_ON_FIRST_FAILURE = bool(_sweep_controls['STOP_ON_FIRST_FAILURE'].value)
SKIP_EXISTING_RUNS = bool(_sweep_controls['SKIP_EXISTING_RUNS'].value)

EXECUTION_MODE = str(_execution_widgets['EXECUTION_MODE'].value).strip()
RUN_SINGLE_MODE = EXECUTION_MODE == 'single_run'
RUN_SWEEP_MODE = EXECUTION_MODE == 'sweep'
COPY_OUTPUTS_TO_DRIVE = bool(_execution_widgets['COPY_OUTPUTS_TO_DRIVE'].value)

_validate_choice('DEVICE', DEVICE, DEVICE_OPTIONS)
_validate_choice('SPLIT_BY', SPLIT_BY, SPLIT_BY_OPTIONS)
_validate_choice('CROSS_ATTENTION_NEIGHBORHOOD', CROSS_ATTENTION_NEIGHBORHOOD, CROSS_ATTENTION_NEIGHBORHOOD_OPTIONS)
_validate_choice('EXECUTION_MODE', EXECUTION_MODE, EXECUTION_MODE_OPTIONS)
_validate_choices('TASK_MODES', TASK_MODES, TASK_MODE_OPTIONS)
_validate_choices('MODEL_PRESETS', MODEL_PRESETS, MODEL_PRESET_OPTIONS)
_validate_choices('NODE_FEATURE_SETS', NODE_FEATURE_SETS, NODE_FEATURE_SET_OPTIONS)
_validate_choices('LR_SCHEDULES', LR_SCHEDULES, LR_SCHEDULE_OPTIONS)
_validate_choices('EC_LABEL_DEPTHS', EC_LABEL_DEPTHS, EC_LABEL_DEPTH_OPTIONS)
_validate_choices('CROSS_ATTENTION_LAYERS', CROSS_ATTENTION_LAYERS, CROSS_ATTENTION_LAYER_OPTIONS)
_validate_choices('CROSS_ATTENTION_HEADS', CROSS_ATTENTION_HEADS, CROSS_ATTENTION_HEAD_OPTIONS)

if EPOCHS < 1:
    raise ValueError('EPOCHS must be at least 1.')
if not 0 < VAL_FRACTION < 1:
    raise ValueError('VAL_FRACTION must be between 0 and 1.')
if any(value <= 0 for value in LEARNING_RATES):
    raise ValueError('Learning rates must be positive.')
if any(value < 0 for value in WEIGHT_DECAYS):
    raise ValueError('Weight decay values cannot be negative.')
if any(value < 1 for value in BATCH_SIZES):
    raise ValueError('Batch sizes must be at least 1.')
if any(value < 0 for value in EC_CONTRASTIVE_WEIGHTS):
    raise ValueError('EC contrastive weights cannot be negative.')
if not 0 <= CROSS_ATTENTION_DROPOUT < 1:
    raise ValueError('CROSS_ATTENTION_DROPOUT must be in the range [0, 1).')
if LR_STEP_SIZE < 1:
    raise ValueError('LR_STEP_SIZE must be at least 1.')
if LR_DECAY_GAMMA <= 0:
    raise ValueError('LR_DECAY_GAMMA must be positive.')
if MAX_SWEEP_RUNS < 1:
    raise ValueError('MAX_SWEEP_RUNS must be at least 1.')

if 'radius_only' in RING_EDGES_MODES and len(RING_EDGES_MODES) == 1:
    REQUIRE_RING_EDGES = False
    PREPARE_MISSING_RING_EDGES = False
elif 'generate_missing_ring' not in RING_EDGES_MODES:
    PREPARE_MISSING_RING_EDGES = False

# Backward-compatible first-value aliases used by command-preview helpers.
TASK_MODE = TASK_MODES[0]
MODEL_PRESET = MODEL_PRESETS[0]
RING_EDGES_MODE = RING_EDGES_MODES[0]
LEARNING_RATE = LEARNING_RATES[0]
WEIGHT_DECAY = WEIGHT_DECAYS[0]
BATCH_SIZE = BATCH_SIZES[0]
SEED = SEEDS[0]
NODE_FEATURE_SET = NODE_FEATURE_SETS[0]
EC_LABEL_DEPTH = EC_LABEL_DEPTHS[0]
EC_CONTRASTIVE_WEIGHT = EC_CONTRASTIVE_WEIGHTS[0]
SINGLE_CROSS_ATTENTION_LAYERS = CROSS_ATTENTION_LAYERS[0]
SINGLE_CROSS_ATTENTION_HEADS = CROSS_ATTENTION_HEADS[0]
LR_SCHEDULE = LR_SCHEDULES[0]

_config_summary = {
    'run_settings': {
        'EPOCHS': EPOCHS,
        'VAL_FRACTION': VAL_FRACTION,
        'DEVICE': DEVICE,
        'RUN_NAME': RUN_NAME or '(auto)',
        'SPLIT_BY': SPLIT_BY,
        'LR_STEP_SIZE': LR_STEP_SIZE,
        'LR_DECAY_GAMMA': LR_DECAY_GAMMA,
        'CROSS_ATTENTION_DROPOUT': CROSS_ATTENTION_DROPOUT,
        'CROSS_ATTENTION_NEIGHBORHOOD': CROSS_ATTENTION_NEIGHBORHOOD,
        'CROSS_ATTENTION_BIDIRECTIONAL': CROSS_ATTENTION_BIDIRECTIONAL,
        'RUN_HELD_OUT_TEST_EVAL': RUN_HELD_OUT_TEST_EVAL,
        'USE_ESM_EMBEDDINGS': USE_ESM_EMBEDDINGS,
        'ESM_EMBEDDINGS_DIR': ESM_EMBEDDINGS_DIR,
        'PREPARE_MISSING_ESM_EMBEDDINGS': PREPARE_MISSING_ESM_EMBEDDINGS,
        'ALLOW_MISSING_ESM_EMBEDDINGS': ALLOW_MISSING_ESM_EMBEDDINGS,
        'ALLOW_MISSING_EXTERNAL_FEATURES': ALLOW_MISSING_EXTERNAL_FEATURES,
        'RING_EXE_PATH': RING_EXE_PATH,
        'RING_EDGE_OUTPUT_DIR': RING_EDGE_OUTPUT_DIR,
        'PRECOMPUTED_RING_EDGES_DIR': PRECOMPUTED_RING_EDGES_DIR,
        'REQUIRE_RING_EDGES': REQUIRE_RING_EDGES,
        'PREPARE_MISSING_RING_EDGES': PREPARE_MISSING_RING_EDGES,
    },
    'experiment_grid': {
        'TASK_MODES': TASK_MODES,
        'MODEL_PRESETS': MODEL_PRESETS,
        'RING_EDGES_MODES': RING_EDGES_MODES,
        'LEARNING_RATES': LEARNING_RATES,
        'WEIGHT_DECAYS': WEIGHT_DECAYS,
        'BATCH_SIZES': BATCH_SIZES,
        'SEEDS': SEEDS,
        'NODE_FEATURE_SETS': NODE_FEATURE_SETS,
        'EC_LABEL_DEPTHS': EC_LABEL_DEPTHS,
        'EC_CONTRASTIVE_WEIGHTS': EC_CONTRASTIVE_WEIGHTS,
        'CROSS_ATTENTION_LAYERS': CROSS_ATTENTION_LAYERS,
        'CROSS_ATTENTION_HEADS': CROSS_ATTENTION_HEADS,
        'LR_SCHEDULES': LR_SCHEDULES,
        'MAX_SWEEP_RUNS': MAX_SWEEP_RUNS,
        'STOP_ON_FIRST_FAILURE': STOP_ON_FIRST_FAILURE,
        'SKIP_EXISTING_RUNS': SKIP_EXISTING_RUNS,
    },
    'execution': {
        'EXECUTION_MODE': EXECUTION_MODE,
        'RUN_SINGLE_MODE': RUN_SINGLE_MODE,
        'RUN_SWEEP_MODE': RUN_SWEEP_MODE,
        'COPY_OUTPUTS_TO_DRIVE': COPY_OUTPUTS_TO_DRIVE,
    },
}

print('Final configuration parsed from grid widget values:')
print(json.dumps(_config_summary, indent=2, sort_keys=True))


## 8. Build Commands and Planned Experiments

Review the first planned grid command and the full planned experiment table before running. The helper functions below build CLI commands only; model training still happens exclusively in `src/train.py`.


In [ ]:
import csv
import hashlib
import itertools
import json
import os
import re
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

local_runs_dir.mkdir(parents=True, exist_ok=True)

VALID_RING_EDGE_MODES = {'radius_only', 'radius_plus_precomputed_ring', 'generate_missing_ring'}


def slugify(value):
    text = str(value).lower().replace('+', 'plus')
    text = re.sub(r'[^a-z0-9]+', '_', text).strip('_')
    return text or 'value'


def format_float_token(value):
    token = f'{float(value):.0e}' if abs(float(value)) < 0.001 else f'{float(value):g}'
    return token.replace('+', '').replace('-', 'm').replace('.', 'p')


def config_hash(config):
    payload = json.dumps(config, sort_keys=True, default=str)
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:8]


def command_string(cmd):
    return ' '.join(shlex.quote(str(part)) for part in cmd)


def make_safe_run_name(config, requested_name=None):
    if requested_name:
        return slugify(requested_name)
    parts = [
        slugify(config['task_mode']),
        slugify(config['model_preset']),
        f"edges{slugify(config['ring_edges_mode'])}",
        f"lr{format_float_token(config['learning_rate'])}",
        f"wd{format_float_token(config['weight_decay'])}",
        f"bs{config['batch_size']}",
        f"seed{config['seed']}",
        f"node{slugify(config['node_feature_set'])}",
    ]
    if config.get('fusion_mode'):
        parts.insert(2, slugify(config['fusion_mode']))
    if config['task'] == 'ec':
        parts.extend([
            f"ecdepth{config['ec_label_depth']}",
            f"eccont{format_float_token(config['ec_contrastive_weight'])}",
        ])
    if config.get('fusion_mode') == 'cross_modal_attention':
        parts.extend([
            f"xal{config['cross_attention_layers']}",
            f"xah{config['cross_attention_heads']}",
        ])
    parts.append(f"h{config_hash(config)}")
    return '_'.join(parts)


def path_from_control(value, *, base=REPO_ROOT):
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path
    return (Path(base) / path).resolve()


def resolve_ring_exe_control(value):
    raw_path = Path(str(value)).expanduser()
    if not str(raw_path):
        return raw_path
    if raw_path.is_absolute():
        return raw_path
    candidates = [
        (REPO_ROOT / raw_path).resolve(),
        (DATA_DIR / raw_path).resolve(),
        (DATA_DIR.parent / raw_path).resolve(),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def sample_files(root, pattern, limit=5):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(pattern))[:limit]


def count_files(root, pattern):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(1 for _ in root.rglob(pattern))


def build_run_config(
    *,
    task_mode,
    model_preset,
    ring_edges_mode,
    learning_rate,
    weight_decay,
    batch_size,
    seed,
    node_feature_set,
    ec_label_depth,
    ec_contrastive_weight,
    cross_attention_layers,
    cross_attention_heads,
    lr_schedule,
    requested_name=None,
):
    ring_edges_mode = normalize_ring_edges_mode(ring_edges_mode)
    if ring_edges_mode not in VALID_RING_EDGE_MODES:
        raise ValueError(f'Unsupported RING_EDGES_MODE: {ring_edges_mode}')
    task, selection_metric = task_map[task_mode]
    preset = dict(preset_map[model_preset])
    uses_esm = bool(preset.get('uses_esm'))
    esm_embeddings_dir = str(path_from_control(ESM_EMBEDDINGS_DIR)) if uses_esm else ''
    ring_output_dir = str(path_from_control(RING_EDGE_OUTPUT_DIR))
    precomputed_ring_dir = str(path_from_control(PRECOMPUTED_RING_EDGES_DIR))
    disabled_ring_dir = str(local_runs_dir / '_radius_only_no_ring_edges')
    if ring_edges_mode == 'radius_only':
        ring_features_dir = disabled_ring_dir
    elif ring_edges_mode == 'radius_plus_precomputed_ring':
        ring_features_dir = precomputed_ring_dir
    else:
        ring_features_dir = ring_output_dir
    prepare_ring = bool(PREPARE_MISSING_RING_EDGES and ring_edges_mode == 'generate_missing_ring')
    require_ring = bool(REQUIRE_RING_EDGES and ring_edges_mode != 'radius_only')
    config = {
        'task_mode': task_mode,
        'task': task,
        'selection_metric': selection_metric,
        'model_preset': model_preset,
        'model_architecture': preset['model_architecture'],
        'fusion_mode': preset.get('fusion_mode'),
        'early_esm_dim': preset.get('early_esm_dim'),
        'early_esm_dropout': preset.get('early_esm_dropout'),
        'epochs': int(EPOCHS),
        'batch_size': int(batch_size),
        'learning_rate': float(learning_rate),
        'weight_decay': float(weight_decay),
        'seed': int(seed),
        'val_fraction': float(VAL_FRACTION),
        'device': DEVICE,
        'split_by': SPLIT_BY,
        'node_feature_set': node_feature_set,
        'ec_label_depth': int(ec_label_depth),
        'ec_contrastive_weight': float(ec_contrastive_weight),
        'lr_schedule': lr_schedule,
        'lr_step_size': int(LR_STEP_SIZE),
        'lr_decay_gamma': float(LR_DECAY_GAMMA),
        'cross_attention_layers': int(cross_attention_layers),
        'cross_attention_heads': int(cross_attention_heads),
        'cross_attention_dropout': float(CROSS_ATTENTION_DROPOUT),
        'cross_attention_neighborhood': CROSS_ATTENTION_NEIGHBORHOOD,
        'cross_attention_bidirectional': bool(CROSS_ATTENTION_BIDIRECTIONAL),
        'run_test_eval': bool(RUN_HELD_OUT_TEST_EVAL),
        'uses_esm': uses_esm,
        'use_esm_embeddings': bool(USE_ESM_EMBEDDINGS and uses_esm),
        'esm_embeddings_dir': esm_embeddings_dir,
        'prepare_missing_esm_embeddings': bool(PREPARE_MISSING_ESM_EMBEDDINGS and uses_esm),
        'allow_missing_esm_embeddings': bool(ALLOW_MISSING_ESM_EMBEDDINGS or not uses_esm),
        'allow_missing_external_features': bool(ALLOW_MISSING_EXTERNAL_FEATURES),
        'ring_edges_mode': ring_edges_mode,
        'use_ring_edges': bool(ring_edges_mode != 'radius_only'),
        'require_ring_edges': require_ring,
        'prepare_missing_ring_edges': prepare_ring,
        'ring_exe_path': str(resolve_ring_exe_control(RING_EXE_PATH)) if ring_edges_mode == 'generate_missing_ring' else '',
        'ring_edge_output_dir': ring_output_dir,
        'precomputed_ring_edges_dir': precomputed_ring_dir if ring_edges_mode == 'radius_plus_precomputed_ring' else '',
        'ring_features_dir': ring_features_dir,
    }
    config['run_name'] = make_safe_run_name(config, requested_name=requested_name)
    config['run_dir'] = local_runs_dir / config['run_name']
    return config


def esm_readiness_issues(config):
    if not config['uses_esm']:
        return []
    issues = []
    embeddings_dir = Path(config['esm_embeddings_dir'])
    embedding_count = count_files(embeddings_dir, '*_esmc.pt')
    if not config['use_esm_embeddings'] and not config['allow_missing_esm_embeddings']:
        issues.append('Selected ESM preset has USE_ESM_EMBEDDINGS=False. Set ALLOW_MISSING_ESM_EMBEDDINGS=True to train explicitly without embeddings.')
    if not config['prepare_missing_esm_embeddings'] and not config['allow_missing_esm_embeddings']:
        if not embeddings_dir.exists():
            issues.append(f'ESM embeddings directory does not exist: {embeddings_dir}')
        elif embedding_count == 0:
            issues.append(f'No *_esmc.pt files found under ESM embeddings directory: {embeddings_dir}')
    return issues


def ring_readiness_issues(config):
    issues = []
    mode = config['ring_edges_mode']
    if mode == 'radius_only':
        return issues
    ring_dir = Path(config['ring_features_dir'])
    ring_count = count_files(ring_dir, '*ringEdges')
    if mode == 'radius_plus_precomputed_ring' and config['require_ring_edges']:
        if not ring_dir.exists():
            issues.append(f'Precomputed RING edge directory does not exist: {ring_dir}')
        elif ring_count == 0:
            issues.append(f'No *ringEdges files found under precomputed RING edge directory: {ring_dir}')
    if mode == 'generate_missing_ring' and config['prepare_missing_ring_edges']:
        ring_exe = Path(config['ring_exe_path'])
        if not ring_exe.is_file():
            issues.append(f'RING executable does not exist: {ring_exe}')
        elif not os.access(ring_exe, os.X_OK):
            issues.append(f'RING executable is not executable: {ring_exe}. Run: chmod +x {ring_exe}')
    return issues


def readiness_issues(config):
    return esm_readiness_issues(config) + ring_readiness_issues(config)


def validate_run_before_training(config):
    issues = readiness_issues(config)
    if issues:
        raise RuntimeError('Pre-training configuration check failed:\n- ' + '\n- '.join(issues))


def print_run_checks(config, *, label):
    embeddings_dir = Path(config['esm_embeddings_dir']) if config['esm_embeddings_dir'] else None
    embedding_files = sample_files(embeddings_dir, '*_esmc.pt') if embeddings_dir is not None else []
    ring_features_dir = Path(config['ring_features_dir'])
    ring_files = sample_files(ring_features_dir, '*ringEdges')
    ring_exe_user_path = path_from_control(RING_EXE_PATH)
    ring_exe = Path(config['ring_exe_path']) if config['ring_exe_path'] else resolve_ring_exe_control(RING_EXE_PATH)
    issues = readiness_issues(config)
    print(f'{label} checks')
    print('  selected model preset:', config['model_preset'])
    print('  selected model uses ESM:', config['uses_esm'])
    print('  ESM embeddings directory:', config['esm_embeddings_dir'] or 'not used by this preset')
    print('  ESM embeddings directory exists:', embeddings_dir.exists() if embeddings_dir is not None else False)
    print('  existing ESM embedding files:', count_files(embeddings_dir, '*_esmc.pt') if embeddings_dir is not None else 0)
    print('  sample ESM embedding files:', [str(path) for path in embedding_files[:3]])
    print('  missing ESM embeddings will be generated:', config['prepare_missing_esm_embeddings'])
    print('  missing ESM embeddings are allowed:', config['allow_missing_esm_embeddings'])
    print('  --esm-embeddings-dir is being passed:', config['uses_esm'])
    print('  --allow-missing-esm-embeddings is being passed:', config['allow_missing_esm_embeddings'])
    print('  --no-prepare-missing-esm-embeddings is being passed:', (not config['prepare_missing_esm_embeddings']) or not config['uses_esm'])
    print('  RING edge mode:', ring_edges_mode_label(config['ring_edges_mode']), f"[{config['ring_edges_mode']}]")
    print('  RING_EXE_PATH user setting:', str(ring_exe_user_path))
    print('  RING_EXE_PATH resolved:', str(ring_exe))
    print('  RING_EXE_PATH exists:', ring_exe.is_file())
    print('  RING_EXE_PATH executable:', os.access(ring_exe, os.X_OK) if ring_exe.exists() else False)
    print('  RING edge files expected under:', ring_features_dir)
    print('  --use-ring-edges is being passed:', config['use_ring_edges'])
    print('  generated RING edge files will be written under:', config['ring_edge_output_dir'])
    print('  existing RING edge files:', count_files(ring_features_dir, '*ringEdges'))
    print('  sample RING edge files:', [str(path) for path in ring_files[:3]])
    print('  precomputed RING edge files are being used:', config['ring_edges_mode'] == 'radius_plus_precomputed_ring')
    print('  missing RING edges will be generated:', config['prepare_missing_ring_edges'])
    print('  --require-ring-edges is being passed:', config['require_ring_edges'])
    print('  --prepare-missing-ring-edges is being passed:', config['prepare_missing_ring_edges'])
    if issues:
        print('  pre-training warnings/errors:')
        for issue in issues:
            print('   -', issue)


def build_train_command(config):
    cmd = [
        sys.executable, str(repo_dir / 'src' / 'train.py'),
        '--task', config['task'],
        '--structure-dir', str(train_dir),
        '--summary-csv', str(train_csv),
        '--runs-dir', str(local_runs_dir),
        '--run-name', config['run_name'],
        '--model-architecture', config['model_architecture'],
        '--epochs', str(config['epochs']),
        '--batch-size', str(config['batch_size']),
        '--learning-rate', str(config['learning_rate']),
        '--weight-decay', str(config['weight_decay']),
        '--seed', str(config['seed']),
        '--val-fraction', str(config['val_fraction']),
        '--device', config['device'],
        '--split-by', config['split_by'],
        '--selection-metric', config['selection_metric'],
        '--node-feature-set', config['node_feature_set'],
        '--lr-schedule', config['lr_schedule'],
    ]
    if config['lr_schedule'] == 'step':
        cmd.extend(['--lr-step-size', str(config['lr_step_size']), '--lr-decay-gamma', str(config['lr_decay_gamma'])])
    if config.get('fusion_mode'):
        cmd.extend(['--fusion-mode', config['fusion_mode']])
    if config.get('fusion_mode') == 'early_fusion':
        cmd.extend(['--use-early-esm', '--early-esm-dim', str(config['early_esm_dim']), '--early-esm-dropout', str(config['early_esm_dropout'])])
    if config.get('fusion_mode') == 'cross_modal_attention':
        cmd.extend([
            '--cross-attention-layers', str(config['cross_attention_layers']),
            '--cross-attention-heads', str(config['cross_attention_heads']),
            '--cross-attention-dropout', str(config['cross_attention_dropout']),
            '--cross-attention-neighborhood', config['cross_attention_neighborhood'],
        ])
        if config['cross_attention_bidirectional']:
            cmd.append('--cross-attention-bidirectional')
    if config['task'] == 'ec':
        cmd.extend(['--ec-label-depth', str(config['ec_label_depth'])])
        cmd.extend(['--ec-contrastive-weight', str(config['ec_contrastive_weight'])])
    if config['run_test_eval']:
        cmd.extend(['--test-structure-dir', str(test_dir), '--test-summary-csv', str(test_csv), '--run-test-eval'])
    if config['uses_esm']:
        cmd.extend(['--esm-embeddings-dir', config['esm_embeddings_dir']])
        if config['allow_missing_esm_embeddings']:
            cmd.append('--allow-missing-esm-embeddings')
        if not config['prepare_missing_esm_embeddings']:
            cmd.append('--no-prepare-missing-esm-embeddings')
    else:
        cmd.extend(['--allow-missing-esm-embeddings', '--no-prepare-missing-esm-embeddings'])
    if config['allow_missing_external_features']:
        cmd.append('--allow-missing-external-features')
    if config['use_ring_edges']:
        cmd.append('--use-ring-edges')
    if config['require_ring_edges']:
        cmd.append('--require-ring-edges')
    if config['prepare_missing_ring_edges']:
        cmd.append('--prepare-missing-ring-edges')
    return cmd


def stream_command(cmd, *, cwd, env):
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return process.wait()


SWEEP_STATUS_COLUMNS = [
    'run_index', 'total_runs', 'run_name', 'task', 'task_mode', 'model_preset',
    'model_architecture', 'fusion_mode', 'learning_rate', 'weight_decay',
    'batch_size', 'seed', 'node_feature_set', 'ec_label_depth',
    'ec_contrastive_weight', 'cross_attention_layers', 'cross_attention_heads',
    'lr_schedule', 'selection_metric', 'val_fraction', 'split_by', 'uses_esm',
    'esm_embeddings_dir', 'prepare_missing_esm_embeddings', 'allow_missing_esm_embeddings',
    'ring_edges_mode', 'use_ring_edges', 'require_ring_edges', 'prepare_missing_ring_edges',
    'ring_exe_path', 'ring_edge_output_dir', 'ring_features_dir',
    'status', 'start_time', 'end_time', 'update_time', 'run_dir',
    'selected_checkpoint', 'selected_metric_value', 'test_summary', 'command', 'error_message',
]


def write_sweep_status(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=SWEEP_STATUS_COLUMNS)
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, '') for column in SWEEP_STATUS_COLUMNS})


def summarize_completed_run(run_dir):
    def read_json(path):
        try:
            return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
        except Exception as exc:
            return {'_read_error': str(exc)}
    run_config = read_json(run_dir / 'run_config.json')
    run_metadata = read_json(run_dir / 'run_metadata.json')
    test_report = read_json(run_dir / 'test_report.json')
    selected_checkpoint = run_metadata.get('selected_checkpoint') or run_config.get('selected_checkpoint')
    selection_metric = run_metadata.get('selection_metric') or run_config.get('selection_metric')
    selected_metric_value = run_metadata.get('selected_metric_value') or run_config.get('selected_metric_value')
    metrics = test_report.get('metrics', {}) if isinstance(test_report.get('metrics'), dict) else {}
    priority = [
        'test_metal_balanced_acc', 'test_metal_collapsed4_balanced_acc',
        'test_ec_group_balanced_acc', 'test_ec_group_level_1_balanced_acc',
        'test_ec_level_1_balanced_acc', 'test_loss',
    ]
    test_summary = {key: metrics[key] for key in priority if key in metrics}
    return {
        'selected_checkpoint': selected_checkpoint,
        'selection_metric': selection_metric,
        'selected_metric_value': selected_metric_value,
        'test_summary': test_summary,
    }


def run_report(runs_dir, out_csv, out_figure):
    report_cmd = [
        sys.executable, str(repo_dir / 'src' / 'report_runs.py'),
        '--runs-dir', str(runs_dir),
        '--out-csv', str(out_csv),
        '--out-figure', str(out_figure),
    ]
    print('Report command:')
    print(command_string(report_cmd))
    return stream_command(report_cmd, cwd=repo_dir, env=base_env)


def build_sweep_runs():
    planned = []
    base_product = itertools.product(
        TASK_MODES,
        MODEL_PRESETS,
        RING_EDGES_MODES,
        LEARNING_RATES,
        WEIGHT_DECAYS,
        BATCH_SIZES,
        SEEDS,
        NODE_FEATURE_SETS,
        LR_SCHEDULES,
    )
    seen_names = set()
    for task_mode, model_preset, ring_edges_mode, lr, wd, batch_size, seed, node_features, lr_schedule in base_product:
        ec_depth_values = EC_LABEL_DEPTHS if task_mode == 'ec_prediction' else [EC_LABEL_DEPTH]
        ec_weight_values = EC_CONTRASTIVE_WEIGHTS if task_mode == 'ec_prediction' else [EC_CONTRASTIVE_WEIGHT]
        cross_layer_values = CROSS_ATTENTION_LAYERS if model_preset == 'GVP + cross-modal attention' else [CROSS_ATTENTION_LAYERS[0]]
        cross_head_values = CROSS_ATTENTION_HEADS if model_preset == 'GVP + cross-modal attention' else [CROSS_ATTENTION_HEADS[0]]
        for ec_depth, ec_weight, cross_layers, cross_heads in itertools.product(
            ec_depth_values,
            ec_weight_values,
            cross_layer_values,
            cross_head_values,
        ):
            config = build_run_config(
                task_mode=task_mode,
                model_preset=model_preset,
                ring_edges_mode=ring_edges_mode,
                learning_rate=lr,
                weight_decay=wd,
                batch_size=batch_size,
                seed=seed,
                node_feature_set=node_features,
                ec_label_depth=ec_depth,
                ec_contrastive_weight=ec_weight,
                cross_attention_layers=cross_layers,
                cross_attention_heads=cross_heads,
                lr_schedule=lr_schedule,
            )
            if config['run_name'] in seen_names:
                continue
            seen_names.add(config['run_name'])
            config['command'] = build_train_command(config)
            planned.append(config)
    return planned


def planned_table_rows(planned_runs):
    rows = []
    for index, run in enumerate(planned_runs, start=1):
        rows.append({
            'run_index': index,
            'run_name': run['run_name'],
            'task': run['task'],
            'task_mode': run['task_mode'],
            'model_preset': run['model_preset'],
            'model_architecture': run['model_architecture'],
            'fusion_mode': run.get('fusion_mode') or 'none',
            'ring_edges_mode': run['ring_edges_mode'],
            'uses_esm': run['uses_esm'],
            'esm_embeddings_dir': run['esm_embeddings_dir'] or 'NA',
            'prepare_missing_esm_embeddings': run['prepare_missing_esm_embeddings'],
            'allow_missing_esm_embeddings': run['allow_missing_esm_embeddings'],
            'use_ring_edges': run['use_ring_edges'],
            'require_ring_edges': run['require_ring_edges'],
            'prepare_missing_ring_edges': run['prepare_missing_ring_edges'],
            'ring_edge_output_dir': run['ring_edge_output_dir'],
            'learning_rate': run['learning_rate'],
            'weight_decay': run['weight_decay'],
            'batch_size': run['batch_size'],
            'seed': run['seed'],
            'node_feature_set': run['node_feature_set'],
            'ec_label_depth': run['ec_label_depth'] if run['task'] == 'ec' else 'NA',
            'ec_contrastive_weight': run['ec_contrastive_weight'] if run['task'] == 'ec' else 'NA',
            'cross_attention_layers': run['cross_attention_layers'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
            'cross_attention_heads': run['cross_attention_heads'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
            'selection_metric': run['selection_metric'],
            'run_dir': str(run['run_dir']),
        })
    return rows


def status_row_for_run(index, total_runs, run):
    return {
        'run_index': index,
        'total_runs': total_runs,
        'run_name': run['run_name'],
        'task': run['task'],
        'task_mode': run['task_mode'],
        'model_preset': run['model_preset'],
        'model_architecture': run['model_architecture'],
        'fusion_mode': run.get('fusion_mode') or 'none',
        'learning_rate': run['learning_rate'],
        'weight_decay': run['weight_decay'],
        'batch_size': run['batch_size'],
        'seed': run['seed'],
        'node_feature_set': run['node_feature_set'],
        'ec_label_depth': run['ec_label_depth'] if run['task'] == 'ec' else 'NA',
        'ec_contrastive_weight': run['ec_contrastive_weight'] if run['task'] == 'ec' else 'NA',
        'cross_attention_layers': run['cross_attention_layers'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
        'cross_attention_heads': run['cross_attention_heads'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
        'lr_schedule': run['lr_schedule'],
        'selection_metric': run['selection_metric'],
        'val_fraction': run['val_fraction'],
        'split_by': run['split_by'],
        'uses_esm': run['uses_esm'],
        'esm_embeddings_dir': run['esm_embeddings_dir'] or 'NA',
        'prepare_missing_esm_embeddings': run['prepare_missing_esm_embeddings'],
        'allow_missing_esm_embeddings': run['allow_missing_esm_embeddings'],
        'ring_edges_mode': run['ring_edges_mode'],
        'use_ring_edges': run['use_ring_edges'],
        'require_ring_edges': run['require_ring_edges'],
        'prepare_missing_ring_edges': run['prepare_missing_ring_edges'],
        'ring_exe_path': run['ring_exe_path'] if run['prepare_missing_ring_edges'] else 'NA',
        'ring_edge_output_dir': run['ring_edge_output_dir'],
        'ring_features_dir': run['ring_features_dir'],
        'status': 'planned',
        'start_time': '',
        'end_time': '',
        'update_time': datetime.now().isoformat(timespec='seconds'),
        'run_dir': str(run['run_dir']),
        'selected_checkpoint': '',
        'selected_metric_value': '',
        'test_summary': '',
        'command': command_string(run['command']),
        'error_message': '',
    }


def mark_status(row, status):
    row['status'] = status
    row['update_time'] = datetime.now().isoformat(timespec='seconds')


planned_sweep_runs = build_sweep_runs()
if not planned_sweep_runs:
    raise ValueError('The experiment grid produced no planned runs. Select at least one value in every grid field.')
if len(planned_sweep_runs) > MAX_SWEEP_RUNS:
    raise ValueError(f'Experiment grid would run {len(planned_sweep_runs)} experiments, above MAX_SWEEP_RUNS={MAX_SWEEP_RUNS}.')

if RUN_NAME and len(planned_sweep_runs) == 1:
    single_named_run = dict(planned_sweep_runs[0])
    single_named_run['run_name'] = slugify(RUN_NAME)
    single_named_run['run_dir'] = local_runs_dir / single_named_run['run_name']
    single_named_run['command'] = build_train_command(single_named_run)
    planned_sweep_runs[0] = single_named_run
elif RUN_NAME and len(planned_sweep_runs) > 1:
    print('Run name is ignored because the experiment grid has more than one planned run; grid run names stay auto-generated.')

single_run_config = planned_sweep_runs[0]
train_cmd = single_run_config['command']
target_run_dir = single_run_config['run_dir']
RUN_NAME = single_run_config['run_name']

def resolve_execution_mode(mode, planned_count):
    if mode == 'auto':
        return 'sweep' if planned_count > 1 else 'single_run'
    return mode

EFFECTIVE_EXECUTION_MODE = resolve_execution_mode(EXECUTION_MODE, len(planned_sweep_runs))
RUN_SINGLE_MODE = EFFECTIVE_EXECUTION_MODE == 'single_run'
RUN_SWEEP_MODE = EFFECTIVE_EXECUTION_MODE == 'sweep'

planned_sweep_table = planned_table_rows(planned_sweep_runs)
sweep_status_path = local_runs_dir / 'sweep_status.csv'
final_comparison_csv = local_runs_dir / 'sweep_comparison.csv'
final_comparison_figure = local_runs_dir / 'sweep_comparison.png'
completed_run_dirs = []
failed_run_dirs = []

os.environ['RING_EXE_PATH'] = str(resolve_ring_exe_control(RING_EXE_PATH))
os.environ['EMBEDDINGS_DIR'] = single_run_config['ring_features_dir']
base_env = os.environ.copy()
base_env['PYTHONPATH'] = str(repo_dir / 'src') + os.pathsep + base_env.get('PYTHONPATH', '')


def env_for_run(config):
    run_env = base_env.copy()
    run_env['RING_EXE_PATH'] = config['ring_exe_path'] or str(resolve_ring_exe_control(RING_EXE_PATH))
    run_env['EMBEDDINGS_DIR'] = config['ring_features_dir']
    return run_env


print_run_checks(single_run_config, label='First planned grid run')
print('\nFirst planned grid command:')
print(command_string(train_cmd))
print('First planned grid output directory:', target_run_dir)
print('\nTotal planned grid runs:', len(planned_sweep_runs))
print('Requested execution mode:', EXECUTION_MODE)
print('Effective execution mode:', EFFECTIVE_EXECUTION_MODE)
print('Sweep status path:', sweep_status_path)
print('Sweep comparison CSV:', final_comparison_csv)
print('Sweep output directory:', local_runs_dir)
if pd is not None:
    display(pd.DataFrame(planned_sweep_table))
else:
    for row in planned_sweep_table:
        print(row)


## 9. Run Single Training or Sweep

This is the only section that starts training. Use `EXECUTION_MODE` to choose one behavior: `preview_only`, `single_run`, `sweep`, or `auto`. In `auto`, one planned sweep combination runs as a single run; more than one planned combination runs as a sweep.

In [ ]:
#@title Execute training or sweep
EXECUTION_MODE = globals().get('EXECUTION_MODE', 'preview_only')  #@param ['preview_only', 'single_run', 'sweep', 'auto']
EFFECTIVE_EXECUTION_MODE = resolve_execution_mode(EXECUTION_MODE, len(planned_sweep_runs))
RUN_SINGLE_MODE = EFFECTIVE_EXECUTION_MODE == 'single_run'
RUN_SWEEP_MODE = EFFECTIVE_EXECUTION_MODE == 'sweep'

print('Requested execution mode:', EXECUTION_MODE)
print('Planned grid runs:', len(planned_sweep_runs))
print('Effective execution mode:', EFFECTIVE_EXECUTION_MODE)
if (RUN_SINGLE_MODE or RUN_SWEEP_MODE) and DEVICE == 'cuda':
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('DEVICE is cuda, but torch.cuda.is_available() is False. Change DEVICE to cpu or enable a GPU runtime.')
if not RUN_SINGLE_MODE and not RUN_SWEEP_MODE:
    print('Training skipped. Set EXECUTION_MODE to single_run, sweep, or auto when ready.')

if RUN_SINGLE_MODE:
    validate_run_before_training(single_run_config)
    print('=' * 60)
    print('Starting first planned grid run')
    print('Run name:', single_run_config['run_name'])
    print('Task:', single_run_config['task'])
    print('Model:', single_run_config['model_preset'])
    print('RING edge mode:', single_run_config['ring_edges_mode'])
    print('Uses ESM:', single_run_config['uses_esm'])
    print('Learning rate:', single_run_config['learning_rate'])
    print('Selection metric:', single_run_config['selection_metric'])
    print('Output directory:', single_run_config['run_dir'])
    print('=' * 60)
    return_code = stream_command(single_run_config['command'], cwd=repo_dir, env=env_for_run(single_run_config))
    if return_code != 0:
        failed_run_dirs.append(single_run_config['run_dir'])
        print(f'Single run failed with exit code {return_code}: {single_run_config["run_dir"]}')
    else:
        completed_run_dirs.append(single_run_config['run_dir'])
        summary = summarize_completed_run(single_run_config['run_dir'])
        print('Single run succeeded:', single_run_config['run_dir'])
        print('Selected checkpoint:', summary.get('selected_checkpoint'))
        print('Best validation metric:', summary.get('selection_metric'), summary.get('selected_metric_value'))
        print('Held-out test metric summary:', summary.get('test_summary'))

if RUN_SWEEP_MODE:
    sweep_status_rows = []
    total_runs = len(planned_sweep_runs)
    for index, run in enumerate(planned_sweep_runs, start=1):
        sweep_status_rows.append(status_row_for_run(index, total_runs, run))
    write_sweep_status(sweep_status_rows, sweep_status_path)

    print('Total planned runs:', total_runs)
    print('Sweep status CSV:', sweep_status_path)
    print('Sweep outputs directory:', local_runs_dir)
    if pd is not None:
        display(pd.DataFrame(planned_table_rows(planned_sweep_runs)))
    else:
        for row in planned_table_rows(planned_sweep_runs):
            print(row)

    for index, run in enumerate(planned_sweep_runs, start=1):
        validate_run_before_training(run)
        status_row = sweep_status_rows[index - 1]
        if SKIP_EXISTING_RUNS and run['run_dir'].exists():
            summary = summarize_completed_run(run['run_dir'])
            mark_status(status_row, 'skipped')
            status_row['end_time'] = datetime.now().isoformat(timespec='seconds')
            status_row['selected_checkpoint'] = summary.get('selected_checkpoint') or ''
            status_row['selected_metric_value'] = '' if summary.get('selected_metric_value') is None else summary.get('selected_metric_value')
            status_row['test_summary'] = json.dumps(summary.get('test_summary') or {}, sort_keys=True)
            status_row['error_message'] = 'run directory already exists'
            write_sweep_status(sweep_status_rows, sweep_status_path)
            completed_run_dirs.append(run['run_dir'])
            print(f'Skipping existing run directory: {run["run_dir"]}')
            continue

        print('=' * 60)
        print(f'Starting run {index} / {total_runs}')
        print('Run name:', run['run_name'])
        print('Task:', run['task'])
        print('Model:', run['model_preset'])
        print('RING edge mode:', run['ring_edges_mode'])
        print('Uses ESM:', run['uses_esm'])
        print('Learning rate:', run['learning_rate'])
        print('Selection metric:', run['selection_metric'])
        print('Output directory:', run['run_dir'])
        print('=' * 60)

        mark_status(status_row, 'running')
        status_row['start_time'] = datetime.now().isoformat(timespec='seconds')
        write_sweep_status(sweep_status_rows, sweep_status_path)
        return_code = stream_command(run['command'], cwd=repo_dir, env=env_for_run(run))
        status_row['end_time'] = datetime.now().isoformat(timespec='seconds')
        if return_code == 0:
            mark_status(status_row, 'succeeded')
            completed_run_dirs.append(run['run_dir'])
            summary = summarize_completed_run(run['run_dir'])
            status_row['selected_checkpoint'] = summary.get('selected_checkpoint') or ''
            status_row['selected_metric_value'] = '' if summary.get('selected_metric_value') is None else summary.get('selected_metric_value')
            status_row['test_summary'] = json.dumps(summary.get('test_summary') or {}, sort_keys=True)
            print(f'Run {index} succeeded:', run['run_dir'])
            print('Selected checkpoint:', summary.get('selected_checkpoint'))
            print('Best validation metric:', summary.get('selection_metric'), summary.get('selected_metric_value'))
            print('Held-out test metric summary:', summary.get('test_summary'))
        else:
            mark_status(status_row, 'failed')
            status_row['error_message'] = f'exit code {return_code}'
            status_row['test_summary'] = ''
            failed_run_dirs.append(run['run_dir'])
            print(f'Run {index} failed with exit code {return_code}: {run["run_dir"]}')
            if STOP_ON_FIRST_FAILURE:
                write_sweep_status(sweep_status_rows, sweep_status_path)
                raise RuntimeError(f'Stopping sweep after failed run {index}: {run["run_name"]}')
        write_sweep_status(sweep_status_rows, sweep_status_path)

    report_code = run_report(local_runs_dir, final_comparison_csv, final_comparison_figure)
    if report_code != 0:
        print(f'Warning: report_runs.py failed with exit code {report_code}')
    succeeded = sum(1 for row in sweep_status_rows if row['status'] == 'succeeded')
    skipped = sum(1 for row in sweep_status_rows if row['status'] == 'skipped')
    failed = sum(1 for row in sweep_status_rows if row['status'] == 'failed')
    print('=' * 60)
    print('Sweep finished')
    print('Succeeded:', succeeded)
    print('Skipped:', skipped)
    print('Failed:', failed)
    print('RING edge modes:', sorted({run['ring_edges_mode'] for run in planned_sweep_runs}))
    print('ESM-enabled planned runs:', sum(1 for run in planned_sweep_runs if run['uses_esm']))
    print('Sweep status CSV:', sweep_status_path)
    print('Final comparison CSV:', final_comparison_csv)
    if RUN_MODE == 'colab':
        print('Copied Drive outputs after running the copy cell:', drive_output_dir)
    else:
        print('Local outputs directory:', OUTPUT_DIR)
    print('=' * 60)


## 10. Copy Run Outputs Back to Drive

This section is Colab-only. In local mode, outputs already remain under `OUTPUT_DIR`.


In [ ]:
#@title Copy outputs to Drive
COPY_OUTPUTS_TO_DRIVE = RUN_MODE == "colab"  #@param {type:"boolean"}

import shutil
from pathlib import Path

if 'completed_run_dirs' not in globals():
    completed_run_dirs = []
if 'failed_run_dirs' not in globals():
    failed_run_dirs = []

drive_runs_dir = drive_output_dir / 'runs'
drive_sweep_dir = drive_output_dir / 'sweeps'
run_dirs_to_copy = []
for candidate in completed_run_dirs + failed_run_dirs:
    candidate = Path(candidate)
    if candidate.exists() and candidate not in run_dirs_to_copy:
        run_dirs_to_copy.append(candidate)
if not run_dirs_to_copy and 'target_run_dir' in globals() and target_run_dir.exists():
    run_dirs_to_copy.append(target_run_dir)

if RUN_MODE == "colab" and COPY_OUTPUTS_TO_DRIVE:
    drive_runs_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for run_dir in run_dirs_to_copy:
        drive_run_dir = drive_runs_dir / run_dir.name
        if drive_run_dir.exists():
            shutil.rmtree(drive_run_dir)
        shutil.copytree(run_dir, drive_run_dir)
        copied.append(drive_run_dir)
        print('Copied run output to:', drive_run_dir)

    drive_sweep_dir.mkdir(parents=True, exist_ok=True)
    for artifact in [sweep_status_path, final_comparison_csv, final_comparison_figure]:
        artifact = Path(artifact)
        if artifact.exists():
            destination = drive_sweep_dir / artifact.name
            shutil.copy2(artifact, destination)
            print('Copied sweep artifact to:', destination)
    if not copied:
        print('No completed run directories found to copy yet.')
    print('Drive output directory:', drive_output_dir)
elif RUN_MODE == "local":
    print('Local mode: Drive copy skipped. Outputs remain under:', OUTPUT_DIR)
else:
    print('Copy skipped.')



## 11. Summarize Runs with `src/report_runs.py`

The summary CSV is the main comparison table. The optional figure is created only when matplotlib is available and the selected validation metric is numeric. Sweep execution already runs this once at the end; this section lets you regenerate the report from local or copied Drive outputs.

In [ ]:
#@title Generate run summary
SUMMARY_SOURCE = 'local_runs' if RUN_MODE == "local" else 'drive_outputs'  #@param ['local_runs', 'drive_outputs']
SUMMARY_BASENAME = 'baseline_first_summary'  #@param {type:"string"}

summary_runs_dir = local_runs_dir if SUMMARY_SOURCE == 'local_runs' else drive_runs_dir
summary_output_dir = OUTPUT_DIR if RUN_MODE == "local" else drive_output_dir
summary_csv = summary_output_dir / f'{SUMMARY_BASENAME}.csv'
summary_figure = summary_output_dir / f'{SUMMARY_BASENAME}.png'
summary_output_dir.mkdir(parents=True, exist_ok=True)

if not summary_runs_dir.exists():
    print(f'No run directory found yet: {summary_runs_dir}')
else:
    report_code = run_report(summary_runs_dir, summary_csv, summary_figure)
    if report_code != 0:
        raise RuntimeError(f'report_runs.py failed with exit code {report_code}')
    print('Summary CSV:', summary_csv)
    print('Summary figure:', summary_figure)



## 12. Display the Summary Table and Figure

Use this table to compare runs by validation-selected metrics first. Test metrics are for final held-out reporting after selecting checkpoints by validation performance.

In [ ]:
from IPython.display import Image, display
import pandas as pd

if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    display(summary_df)
else:
    print(f'Summary CSV not found yet: {summary_csv}')

if summary_figure.exists():
    display(Image(filename=str(summary_figure)))
else:
    print(f'Optional summary figure not found: {summary_figure}')


## 13. Final Model-Selection Checklist

- Confirm every final run used the non-overlapped PinMyMetal split, or clearly label exact/possibly-overlapped runs as secondary reference only.
- Compare Only-GVP, Only-ESM, GVP + late fusion, and GVP + early fusion before moving to complex fusion modes.
- Select checkpoints and model variants by validation metrics, not by repeatedly checking the held-out test set.
- For metal prediction, report both 6-class metrics and collapsed-4 metrics from the selected checkpoint.
- For EC prediction, report supported EC level metrics from the selected checkpoint.
- Preserve `run_config.json`, `run_metadata.json`, `dataset_summary.json`, `test_report.json`, selected checkpoint path, split identity, overlap status, and the summary CSV/figure in Drive.